# Ultra-Hot Jupiter Sector Inspection

Ultra-hot Jupiters: R_b > 0.70 Rjup, P < 3d, FG host (Teff 4800–7500 K), TESS SPOC 2-min.

For each candidate, shows **all available TESS sectors**.
Each panel: faint = full LC, bright = longest continuous gap-free segment (gap < 10 min).
Use this to visually pick the best sector per planet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
import pickle, os, time, warnings
warnings.filterwarnings('ignore')

RJUP_RSUN  = 0.10045
RSUN_AU    = 0.00465047
GAP_MIN    = 10
GAP_DAYS   = GAP_MIN / 60 / 24
COLOR_LC   = '#ef9a9a'   # faint (light red-pink)
COLOR_SEG  = '#b71c1c'   # bright (deep red — hotter than regular HJ)
CACHE_FILE = '../data/uhj_sector_lc_cache.pkl'
OUT_DIR    = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
print('Setup done.')

In [ ]:
coverage = pd.read_csv('../data/tess_coverage_raw.csv')
tepcat   = pd.read_csv('../data/planets_ready_for_modeling.csv')

for col in ['R_b','R_A','a(AU)','Period','Teff']:
    tepcat[col] = pd.to_numeric(tepcat[col], errors='coerce')
tepcat['k']        = tepcat['R_b'] * RJUP_RSUN / tepcat['R_A']
tepcat['aRs']      = tepcat['a(AU)'] / (tepcat['R_A'] * RSUN_AU)
tepcat['depth_pct']= (tepcat['k']**2) * 100
arg = ((1 + tepcat['k']) / tepcat['aRs']).clip(-1, 1)
tepcat['T14_hr']   = (tepcat['Period'] * 24 / np.pi) * np.arcsin(arg)

tepcat2 = tepcat.drop(columns=['has_pdcsap','n_sectors','sector_list'], errors='ignore')
df = tepcat2.merge(coverage[['System','has_pdcsap','n_sectors','sector_list']],
                   on='System', how='left')
df['n_sectors'] = pd.to_numeric(df['n_sectors'], errors='coerce')

# ── UHJ filter ────────────────────────────────────────────────────────────────
uhj_all = df[
    (df['R_b'] > 0.70) &
    (df['Period'] < 3.0) &
    (df['Teff'] >= 4800) & (df['Teff'] <= 7500) &
    (df['has_pdcsap'] == True) &
    (~df['System'].str.contains('WD_', na=False))
].copy()

uhj_all['score']          = uhj_all['depth_pct'] * np.log1p(uhj_all['n_sectors'])
uhj_all['n_transits_est'] = (13.0 / uhj_all['Period']).astype(int)
uhj_all = uhj_all.sort_values('score', ascending=False)

print(f'Total UHJ pool: {len(uhj_all)} planets')
print()

# ── Top 10 candidates (score-ranked) ─────────────────────────────────────────
# Exclude any already selected as regular HJ in selected_10_planets.csv
try:
    selected = pd.read_csv('../selected_10_planets.csv')
    already  = selected['System'].tolist()
except FileNotFoundError:
    already = []

uhj_candidates = uhj_all[~uhj_all['System'].isin(already)].head(10).copy()
uhj_candidates['batch'] = 'UHJ candidate'

print('Top 10 UHJ candidates by score (depth × log(n_sectors)):')
show = ['System','R_b','Period','depth_pct','T14_hr','Teff','n_sectors','n_transits_est','score']
print(uhj_candidates[show].to_string(index=False))

In [ ]:
def parse_sector_list(s):
    if pd.isna(s) or str(s).strip() in ('', '[]'):
        return []
    s = str(s).strip().strip('[]')
    return [int(x.strip()) for x in s.split(',') if x.strip().isdigit()]


def download_lc_normalized(host, sector, retries=3):
    for attempt in range(retries):
        try:
            sr = lk.search_lightcurve(host, mission='TESS', sector=sector,
                                       author='SPOC', exptime=120)
            if sr is None or len(sr) == 0:
                return None
            lc = sr[0].download(flux_column='pdcsap_flux')
            if lc is None:
                return None
            lc  = lc.remove_nans()
            med = np.nanmedian(lc.flux.value)
            if med == 0:
                return None
            return {'time': np.array(lc.time.value),
                    'flux': np.array(lc.flux.value) / med}
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(10)
            else:
                print(f'      FAILED S{sector}: {e}')
    return None


def longest_continuous_segment(tarr, farr, gap_days):
    diffs      = np.diff(tarr)
    gap_idx    = np.where(diffs > gap_days)[0]
    boundaries = np.concatenate([[-1], gap_idx, [len(tarr) - 1]])
    best_len, best_start, best_end = 0, 0, 0
    for i in range(len(boundaries) - 1):
        s = boundaries[i] + 1
        e = boundaries[i + 1]
        seg_len = tarr[e] - tarr[s]
        if seg_len > best_len:
            best_len  = seg_len
            best_start = s
            best_end   = e
    mask = np.zeros(len(tarr), bool)
    mask[best_start: best_end + 1] = True
    return mask, tarr[best_start], tarr[best_end], best_len


print('Helpers defined.')

In [ ]:
# ── Load or build cache ───────────────────────────────────────────────────────
lc_cache = {}

if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, 'rb') as f:
            lc_cache = pickle.load(f)
        print(f'Loaded cache: {len(lc_cache)} entries')
    except (EOFError, Exception) as e:
        print(f'Cache corrupted ({e}) — starting fresh.')
        os.remove(CACHE_FILE)
        lc_cache = {}

for _, row in uhj_candidates.iterrows():
    planet  = row['System']
    host    = row['host_star']
    sectors = parse_sector_list(row['sector_list'])
    if not sectors:
        print(f'{planet}: no sectors listed')
        continue
    for sec in sectors:
        key = (planet, sec)
        if key in lc_cache:
            continue
        print(f'  Downloading {planet} S{sec}...', end=' ', flush=True)
        data = download_lc_normalized(host, sec)
        lc_cache[key] = data
        print(f'{len(data["time"])} pts' if data else 'NO DATA')
        tmp = CACHE_FILE + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(lc_cache, f)
        os.replace(tmp, CACHE_FILE)
        time.sleep(0.3)

print(f'\nCache total: {len(lc_cache)} entries')

In [ ]:
# ── Plot: one block per planet, one panel per sector ─────────────────────────
NCOLS = 5

for _, row in uhj_candidates.iterrows():
    planet  = row['System']
    sectors = parse_sector_list(row['sector_list'])
    rb      = float(row['R_b'])       if pd.notna(row['R_b'])       else float('nan')
    period  = float(row['Period'])    if pd.notna(row['Period'])    else float('nan')
    depth   = float(row['depth_pct']) if pd.notna(row['depth_pct']) else float('nan')
    t14     = float(row['T14_hr'])    if pd.notna(row['T14_hr'])    else float('nan')
    teff    = int(row['Teff'])        if pd.notna(row['Teff'])       else 0
    n_tr_est= int(row['n_transits_est'])

    avail = [s for s in sectors if lc_cache.get((planet, s)) is not None]
    if not avail:
        print(f'{planet}: no data to plot')
        continue

    nrows = int(np.ceil(len(avail) / NCOLS))
    fig, axes = plt.subplots(nrows, NCOLS,
                              figsize=(4.5 * NCOLS, 3.2 * nrows),
                              squeeze=False)
    fig.suptitle(
        f"{planet}  [Ultra-hot Jupiter]  |  R_b={rb:.2f} Rjup  |  P={period:.3f} d  "
        f"|  depth≈{depth:.3f}%  |  T14≈{t14:.2f} hr  |  Teff={teff} K  "
        f"|  ~{n_tr_est} transits/segment  |  {len(avail)} sectors",
        fontsize=11, fontweight='bold'
    )

    for idx, sec in enumerate(avail):
        ax   = axes[idx // NCOLS][idx % NCOLS]
        data = lc_cache[(planet, sec)]
        tarr = data['time']
        farr = data['flux']

        seg_mask, seg_t0, seg_t1, seg_days = \
            longest_continuous_segment(tarr, farr, GAP_DAYS)
        seg_pts    = int(seg_mask.sum())
        n_transits = int(seg_days / period) if period > 0 else 0

        # faint full LC
        ax.plot(tarr, farr, '.', color=COLOR_LC, ms=1.0, alpha=0.25, rasterized=True)
        # bright best segment
        ax.plot(tarr[seg_mask], farr[seg_mask], '.', color=COLOR_SEG,
                ms=1.2, alpha=0.8, rasterized=True)
        # shaded span
        ax.axvspan(seg_t0, seg_t1, alpha=0.07, color=COLOR_SEG)

        ax.set_title(
            f'S{sec}  |  {seg_days:.1f}d  |  {seg_pts} pts  |  ~{n_transits} transits',
            fontsize=8
        )
        ax.set_xlabel('BTJD', fontsize=7)
        ax.tick_params(labelsize=7)
        ax.set_ylabel('')

    for idx in range(len(avail), nrows * NCOLS):
        axes[idx // NCOLS][idx % NCOLS].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fname = planet.replace(' ', '_').replace('/', '_')
    out   = f'{OUT_DIR}/uhj_sectors_{fname}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')